# Qwen3.5-Plus API Direct Evaluation (Task4 + Task6)

- Evaluate the test split directly with Qwen3.5-Plus API
- Inputs include meme image + OCR text
- One API call returns both Task4 and Task6 outputs
- Metrics include hard-hard and soft-soft

In [1]:
import base64
import json
import os
import random
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import List, Tuple

import numpy as np
from openai import OpenAI
from sklearn.metrics import f1_score
from tqdm import tqdm

from dataset import create_data_splits, TASK6_LABELS
from pipeline_b import load_rows

In [2]:
@dataclass
class Config:
    data_dir: str = "/root/QWen/EXIST 2024 Memes Dataset"
    output_dir: str = f"/root/QWen/outputs/qwen35plus_api_direct_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    seed: int = 42
    force_split: bool = False
    max_test_rows: int = 0

    # Evaluation mode: hard-hard / soft-soft / both
    eval_mode: str = "both"

    # Keep this switch for consistent post-processing behavior
    disable_hierarchical_gate: bool = True

    # API settings
    api_model: str = "qwen3.5-plus"
    api_base_url: str = "https://coding.dashscope.aliyuncs.com/v1"
    request_timeout: int = 45

    # Performance settings
    max_workers: int = 4
    max_retries: int = 2
    retry_sleep: float = 1.2
    max_tokens: int = 64

cfg = Config()
random.seed(cfg.seed)
np.random.seed(cfg.seed)

os.makedirs(cfg.output_dir, exist_ok=True)
api_key = os.getenv("DASHSCOPE_API_KEY", "").strip() or "sk-sp-aa135b5506174179b9a77b9b43db1f85"
if not api_key:
    raise ValueError("DASHSCOPE_API_KEY is empty. Please export it first.")

client = OpenAI(api_key=api_key, base_url=cfg.api_base_url, timeout=cfg.request_timeout)
print(cfg)

Config(data_dir='/root/QWen/EXIST 2024 Memes Dataset', output_dir='/root/QWen/outputs/qwen35plus_api_direct_20260322_043359', seed=42, force_split=False, max_test_rows=0, eval_mode='both', disable_hierarchical_gate=True, api_model='qwen3.5-plus', api_base_url='https://coding.dashscope.aliyuncs.com/v1', request_timeout=45, max_workers=4, max_retries=2, retry_sleep=1.2, max_tokens=64)


In [3]:
splits = create_data_splits(cfg.data_dir, seed=cfg.seed, force_split=cfg.force_split)
test_ids = splits.get("test", [])
test_rows = load_rows(cfg.data_dir, test_ids)
if cfg.max_test_rows > 0:
    test_rows = test_rows[: cfg.max_test_rows]

print(f"[data] test rows: {len(test_rows)}")

Loading existing data splits from /root/QWen/EXIST 2024 Memes Dataset/data_splits.json
[data] test rows: 405


In [4]:
def image_to_data_url(image_path: str) -> str:
    ext = Path(image_path).suffix.lower().replace(".", "")
    if ext == "jpg":
        ext = "jpeg"
    if ext not in {"jpeg", "png", "webp", "gif"}:
        ext = "jpeg"
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:image/{ext};base64,{b64}"


def build_joint_prompt(text: str, lang: str) -> str:
    labels_json = json.dumps(TASK6_LABELS, ensure_ascii=False)
    return f"""You are a strict classifier.
Use BOTH the image and OCR text.

Language: {lang}
OCR:
{text}

Task4 definition:
A meme is sexist if it expresses prejudice, contempt, stereotyping, discrimination,
or demeaning attitudes toward women or girls.

Task6 label definitions:
- IDEOLOGICAL-INEQUALITY: supports or normalizes unequal roles, rights, or value between men and women.
- STEREOTYPING-DOMINANCE: reinforces traditional gender stereotypes, male superiority, or dominance over women.
- OBJECTIFICATION: portrays women mainly as bodies, body parts, or sexual objects, rather than as persons.
- SEXUAL-VIOLENCE: depicts, threatens, jokes about, or normalizes sexual coercion, assault, or abuse.
- MISOGYNY-NON-SEXUAL-VIOLENCE: expresses hatred, insults, humiliation, hostility, or non-sexual aggression toward women.

Decision rules:
- Task4: output YES or NO.
- Task6: select zero or more labels only from valid label set.
- If no Task6 label applies, return an empty list.

Return strict JSON only (no markdown, no explanation):
{{
  \"task4\": \"YES\" or \"NO\",
  \"task6_labels\": [\"LABEL1\", \"LABEL2\"]
}}

Valid Task6 labels:
{labels_json}
""".strip()


def parse_joint_output(raw_output: str) -> Tuple[int, List[int], str]:
    raw = (raw_output or "").strip()
    if not raw:
        return 0, [0] * len(TASK6_LABELS), raw

    start = raw.find("{")
    end = raw.rfind("}")
    raw_json = raw[start:end + 1] if (start != -1 and end != -1 and end > start) else raw

    obj = None
    try:
        obj = json.loads(raw_json)
    except Exception:
        obj = None

    if not isinstance(obj, dict):
        task4 = 1 if re.search(r"\bYES\b", raw.upper()) else 0
        pred6 = [0] * len(TASK6_LABELS)
        for i, lab in enumerate(TASK6_LABELS):
            if lab in raw.upper():
                pred6[i] = 1
        return task4, pred6, raw

    task4_raw = str(obj.get("task4", "NO")).strip().upper()
    task4 = 1 if task4_raw == "YES" else 0

    labels = obj.get("task6_labels", [])
    if not isinstance(labels, list):
        labels = []
    labels = [str(x).strip().upper() for x in labels if isinstance(x, str)]
    pred6 = [1 if lab in labels else 0 for lab in TASK6_LABELS]

    return task4, pred6, raw


def call_qwen_joint(row) -> Tuple[int, List[int], str]:
    prompt = build_joint_prompt(row.text, row.lang)
    image_url = image_to_data_url(row.image_path)

    last_err = None
    for attempt in range(cfg.max_retries):
        try:
            resp = client.chat.completions.create(
                model=cfg.api_model,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "image_url", "image_url": {"url": image_url}},
                            {"type": "text", "text": prompt},
                        ],
                    }
                ],
                temperature=0,
                max_tokens=cfg.max_tokens,
                response_format={"type": "json_object"},
            )
            raw = (resp.choices[0].message.content or "").strip()
            pred4, pred6, raw = parse_joint_output(raw)

            if (not cfg.disable_hierarchical_gate) and pred4 == 0:
                pred6 = [0] * len(TASK6_LABELS)

            return pred4, pred6, raw
        except Exception as e:
            last_err = e
            if attempt < cfg.max_retries - 1:
                time.sleep(cfg.retry_sleep * (attempt + 1) + random.uniform(0.0, 0.3))

    err_raw = json.dumps({"error": str(last_err)}, ensure_ascii=False)
    return 0, [0] * len(TASK6_LABELS), err_raw

In [5]:
y4_true, y4_pred = [], []
y6_true, y6_pred = [], []
y4_true_soft, y4_pred_soft = [], []
y6_true_soft, y6_pred_soft = [], []
pred_rows = []


def process_one(row):
    pred4, pred6, raw = call_qwen_joint(row)
    return {
        "sample_id": row.sample_id,
        "task4_true": row.task4,
        "task4_pred": pred4,
        "task4_raw": raw,
        "task6_true": row.task6,
        "task6_pred": pred6,
        "task6_raw": raw,
        "task4_soft_true": float(row.task4_soft),
        "task6_soft_true": [float(x) for x in row.task6_soft],
    }


results = []
with ThreadPoolExecutor(max_workers=cfg.max_workers) as executor:
    futures = [executor.submit(process_one, row) for row in test_rows]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Qwen API evaluating"):
        results.append(fut.result())

result_map = {x["sample_id"]: x for x in results}
ordered_results = [result_map[r.sample_id] for r in test_rows if r.sample_id in result_map]

for row in ordered_results:
    y4_true.append(row["task4_true"])
    y4_pred.append(row["task4_pred"])
    y6_true.append(row["task6_true"])
    y6_pred.append(row["task6_pred"])

    y4_true_soft.append(row["task4_soft_true"])
    y4_pred_soft.append(float(row["task4_pred"]))
    y6_true_soft.append(row["task6_soft_true"])
    y6_pred_soft.append([float(x) for x in row["task6_pred"]])

    pred_rows.append({
        "sample_id": row["sample_id"],
        "task4_true": row["task4_true"],
        "task4_pred": row["task4_pred"],
        "task4_raw": row["task4_raw"],
        "task6_true": row["task6_true"],
        "task6_pred": row["task6_pred"],
        "task6_raw": row["task6_raw"],
    })

y6_true_np = np.array(y6_true)
y6_pred_np = np.array(y6_pred)

metrics = {"num_samples": len(ordered_results), "eval_mode": cfg.eval_mode}
if cfg.eval_mode in ("hard-hard", "both"):
    metrics["hard_hard"] = {
        "task4": {
            "f1_binary": float(f1_score(y4_true, y4_pred, average="binary", zero_division=0)),
            "f1_macro": float(f1_score(y4_true, y4_pred, average="macro", zero_division=0)),
        },
        "task6": {
            "f1_macro": float(f1_score(y6_true_np, y6_pred_np, average="macro", zero_division=0)),
            "f1_micro": float(f1_score(y6_true_np, y6_pred_np, average="micro", zero_division=0)),
            "f1_samples": float(f1_score(y6_true_np, y6_pred_np, average="samples", zero_division=0)),
        },
    }

if cfg.eval_mode in ("soft-soft", "both"):
    y4_true_soft_np = np.array(y4_true_soft, dtype=np.float32)
    y4_pred_soft_np = np.array(y4_pred_soft, dtype=np.float32)
    y6_true_soft_np = np.array(y6_true_soft, dtype=np.float32)
    y6_pred_soft_np = np.array(y6_pred_soft, dtype=np.float32)

    task4_brier = float(np.mean((y4_pred_soft_np - y4_true_soft_np) ** 2))
    task6_brier_per_label = np.mean((y6_pred_soft_np - y6_true_soft_np) ** 2, axis=0)
    task6_brier_macro = float(np.mean(task6_brier_per_label))
    task6_brier_micro = float(np.mean((y6_pred_soft_np - y6_true_soft_np) ** 2))

    metrics["soft_soft"] = {
        "task4": {"brier": task4_brier},
        "task6": {"brier_macro": task6_brier_macro, "brier_micro": task6_brier_micro},
    }

with open(os.path.join(cfg.output_dir, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

with open(os.path.join(cfg.output_dir, "predictions.jsonl"), "w", encoding="utf-8") as f:
    for row in pred_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("\n==== Qwen3.5-Plus API Evaluation done ====")
print(f"[config] model: {cfg.api_model}")
print(f"[config] hierarchical_gate: {not cfg.disable_hierarchical_gate}")
if "hard_hard" in metrics:
    print(f"[hard-hard] Task4 F1(binary): {metrics['hard_hard']['task4']['f1_binary']:.4f}")
    print(f"[hard-hard] Task4 F1(macro):  {metrics['hard_hard']['task4']['f1_macro']:.4f}")
    print(f"[hard-hard] Task6 F1(macro):  {metrics['hard_hard']['task6']['f1_macro']:.4f}")
    print(f"[hard-hard] Task6 F1(micro):  {metrics['hard_hard']['task6']['f1_micro']:.4f}")
    print(f"[hard-hard] Task6 F1(samples):{metrics['hard_hard']['task6']['f1_samples']:.4f}")
if "soft_soft" in metrics:
    print(f"[soft-soft] Task4 Brier:      {metrics['soft_soft']['task4']['brier']:.4f}")
    print(f"[soft-soft] Task6 Brier(macro): {metrics['soft_soft']['task6']['brier_macro']:.4f}")
    print(f"[soft-soft] Task6 Brier(micro): {metrics['soft_soft']['task6']['brier_micro']:.4f}")
print(f"samples: {metrics['num_samples']}")
print(f"metrics path: {os.path.join(cfg.output_dir, 'metrics.json')}")
print(f"predictions path: {os.path.join(cfg.output_dir, 'predictions.jsonl')}")

Qwen API evaluating: 100%|██████████| 405/405 [02:17<00:00,  2.94it/s]


==== Qwen3.5-Plus API Evaluation done ====
[config] model: qwen3.5-plus
[config] hierarchical_gate: False
[hard-hard] Task4 F1(binary): 0.7818
[hard-hard] Task4 F1(macro):  0.7751
[hard-hard] Task6 F1(macro):  0.3049
[hard-hard] Task6 F1(micro):  0.3122
[hard-hard] Task6 F1(samples):0.1220
[soft-soft] Task4 Brier:      0.1661
[soft-soft] Task6 Brier(macro): 0.1050
[soft-soft] Task6 Brier(micro): 0.1050
samples: 405
metrics path: /root/QWen/outputs/qwen35plus_api_direct_20260322_043359/metrics.json
predictions path: /root/QWen/outputs/qwen35plus_api_direct_20260322_043359/predictions.jsonl
